# LATTICE: Graph Self-Supervised Learning for Multimodal Spatial Omics Integration
**ArXivist-generated reproduction notebook**

Paper: [arXiv:2607.14410](https://arxiv.org/abs/2607.14410)
Generated: 2026-07-21

This notebook walks through the key components of the implementation, runs a
small-scale training loop on synthetic data, and shows how the paper's
reported results (Table 2/3) compare to what this reproduction can currently
verify.

> **Data note**: the paper's real evaluation cohort (11 private melanoma
> samples, 54,912 spots) cannot be released. Everything below runs on a
> small SYNTHETIC dataset generated to match the paper's documented shapes
> — see `data/README_data.md` for details.


In [ ]:
# Check Python version, GPU availability, and key dependencies
import sys, torch
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU — training will be slow (the paper itself trained CPU-only)")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# Install the project in editable mode (run once)
import subprocess
result = subprocess.run(["pip", "install", "-e", ".."], capture_output=True, text=True)
print(result.stdout[-2000:] if result.returncode == 0 else result.stderr[-2000:])


## Paper Overview

**Problem.** Spatially resolved omics studies increasingly combine multiple
modalities (RNA, ATAC, histone marks) per tissue spot, but analysis is
usually still done one modality at a time.

**Core idea.** LATTICE concatenates five aligned modality blocks per Visium
spot (Visium RNA, projected scMultiome RNA, projected scMultiome ATAC gene
scores, spatial ATAC, spatial CUT&Tag), builds a spatial k-nearest-neighbor
graph over spots, and trains a **TransformerConv** graph encoder with three
self-supervised objectives:

1. **Masked reconstruction** — hide some feature entries, reconstruct them from the embedding.
2. **Cross-modal alignment** — an NCE-style contrastive loss tying together views of the same spot.
3. **Spatial smoothness** — a graph-Laplacian penalty encouraging nearby spots to have similar embeddings.

**Section map:**
| Paper section | What it covers | Code |
|---|---|---|
| 3.1 | Multimodal input construction (Eq. 1) | `data/dataset.py` |
| 3.2 | Spatial graph construction (Eq. 2-3) | `data/graph_utils.py` |
| 3.3 | Encoder, decoder, losses (Eq. 4-9) | `models/*.py`, `training/losses.py` |
| 3.4 / Algorithm 1 | Training loop | `training/trainer.py` |
| 4 | Evaluation (ARI, NMI, MUS, ...) | `evaluation/metrics.py` |


In [ ]:
import sys
sys.path.insert(0, "../src")
import torch
torch.manual_seed(42)


## Component 1 — Modality Input Adapters + Fusion

Each of the 5 modality blocks $X^{(b)} \in \mathbb{R}^{N \times d_b}$ is projected
into a shared hidden space via a modality-specific linear adapter, then fused
by presence-weighted mean pooling (Appendix H):

$$X = [X^{(1)}, X^{(2)}, \ldots, X^{(B)}] \in \mathbb{R}^{N \times D}$$


In [ ]:
from lattice.models.modality_adapters import ModalityInputAdapter, ModalityAwareFusion

hidden_dim = 128
num_modalities = 5
n_spots = 16  # toy batch of 16 spots for demo

adapters = [ModalityInputAdapter(in_dim=64, hidden_dim=hidden_dim) for _ in range(num_modalities)]
fusion = ModalityAwareFusion(num_modalities=num_modalities)

toy_blocks = [torch.randn(n_spots, 64) for _ in range(num_modalities)]
presence_mask = (torch.rand(n_spots, num_modalities) > 0.1).float()

adapted = [adapter(block) for adapter, block in zip(adapters, toy_blocks)]
fused = fusion(adapted, presence_mask)

print(f"Input shape (per modality):  {toy_blocks[0].shape}")
print(f"Adapted shape (per modality): {adapted[0].shape}")
print(f"Fused output shape: {fused.shape}")
print(f"Expected: [16, {hidden_dim}]")


## Component 2 — Spatial Graph + TransformerConv Encoder

A k-NN graph ($k=6$) is built over spot coordinates (Eq. 2), then a stack of
3 TransformerConv layers (4 heads, dropout 0.1, LayerNorm, ReLU) produces the
spot embeddings $Z$:

$$\mathcal{N}(i) = \mathrm{kNN}(s_i), \qquad Z = f_\theta(X, G)$$


In [ ]:
from lattice.models.graph_encoder import SpatialGraphBuilder, LatticeGraphEncoder

coords = torch.rand(n_spots, 2) * 10  # toy spatial coordinates
graph_builder = SpatialGraphBuilder(k=4, edge_weight_mode="uniform")  # k=4 since we only have 16 toy spots
edge_index, edge_weight = graph_builder.build(coords)

encoder = LatticeGraphEncoder(hidden_dim=hidden_dim, num_layers=3, num_heads=4, dropout=0.1)
z = encoder(fused, edge_index, edge_weight)

print(f"Graph: {edge_index.shape[1]} directed edges over {n_spots} nodes")
print(f"Embedding shape: {z.shape}")
print(f"Expected: [16, {hidden_dim}]")


## Component 3 — Decoder + Cross-Modal Projection Heads

The decoder reconstructs the full feature matrix from $Z$ (Eq. 5); two
projection heads produce $h^{(a)}, h^{(b)}$ used in the cross-modal
alignment loss (Eq. 7-8).


In [ ]:
from lattice.models.decoder import ReconstructionDecoder
from lattice.models.projection_heads import ModalityProjectionHead

total_feature_dim = 64 * num_modalities  # D = sum of modality block dims
decoder = ReconstructionDecoder(hidden_dim=hidden_dim, output_dim=total_feature_dim, decoder_hidden_width=256)
proj_head_a = ModalityProjectionHead(hidden_dim=hidden_dim, proj_hidden_width=64, output_dim=64)
proj_head_b = ModalityProjectionHead(hidden_dim=hidden_dim, proj_hidden_width=64, output_dim=64)

x_hat = decoder(z)
h_a = proj_head_a(z)
h_b = proj_head_b(z)

print(f"Reconstruction shape: {x_hat.shape}  (expected [16, {total_feature_dim}])")
print(f"Projection h_a shape: {h_a.shape}  (expected [16, 64])")
print(f"Projection h_b shape: {h_b.shape}  (expected [16, 64])")


## Component 4 — Losses (Eq. 6-10)

- **Masked reconstruction**: Huber loss on hidden entries (Appendix H; default — see README for the Eq.6-vs-Appendix-H ambiguity).
- **Cross-modal alignment**: NCE-style contrastive loss (Eq. 8).
- **Spatial smoothness**: graph-Laplacian quadratic form (Eq. 9).

$$\mathcal{L} = \lambda_1 \mathcal{L}_{rec} + \lambda_2 \mathcal{L}_{align} + \lambda_3 \mathcal{L}_{spatial}$$


In [ ]:
from lattice.training.losses import combined_loss
from lattice.training.trainer import sample_mask

x = torch.cat(toy_blocks, dim=1)
mask = sample_mask(x.shape, ratio=0.15, device=x.device)

outputs = {
    "x": x, "x_hat": x_hat, "mask": mask,
    "h_a": h_a, "h_b": h_b, "z": z,
    "edge_index": edge_index, "edge_weight": edge_weight,
}
losses = combined_loss(outputs, lambda_rec=1.0, lambda_align=0.5, lambda_spatial=0.1,
                        recon_loss="huber", recon_loss_delta=1.0, alignment_temperature=0.1)
for k, v in losses.items():
    print(f"{k}: {float(v.detach()):.4f}")


## Mini-Training Demo (synthetic data, ~10 steps)

This uses a tiny synthetic dataset (2 samples, 100 spots each) so it runs in
seconds on CPU with no downloads.


In [ ]:
from lattice.data.dataset import SyntheticSpatialMultimodalDataset

mini_dataset = SyntheticSpatialMultimodalDataset(
    num_samples=2, spots_per_sample=100, gene_count_range=(32, 32),
    num_modality_blocks=5, seed=0,
)
print(f"Mini dataset: {len(mini_dataset)} samples, modality_dims={mini_dataset.modality_dims}")
sample0 = mini_dataset[0]
print({k: (v.shape if hasattr(v, 'shape') else type(v)) for k, v in sample0.items()})


In [ ]:
from lattice.models.lattice_model import LatticeModel

mini_model = LatticeModel(
    modality_dims=mini_dataset.modality_dims,
    hidden_dim=32, graph_num_layers=2, graph_num_heads=4, graph_dropout=0.1,
    decoder_hidden_width=64, proj_hidden_width=32, proj_output_dim=16,
    knn_k=6, edge_weight_mode="uniform",
)
num_params = sum(p.numel() for p in mini_model.parameters())
print(f"Mini model parameter count: {num_params:,}")


In [ ]:
optimizer = torch.optim.AdamW(mini_model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_history = []

for step in range(10):
    sample = mini_dataset[step % len(mini_dataset)]
    x = torch.cat(sample["modality_blocks"], dim=1)
    mask_step = sample_mask(x.shape, ratio=0.15, device=x.device)

    optimizer.zero_grad()
    out = mini_model(sample["modality_blocks"], sample["presence_mask"], sample["coords"], mask_step)
    step_losses = combined_loss(out, recon_loss="huber", recon_loss_delta=1.0)
    step_losses["total"].backward()
    optimizer.step()
    loss_history.append(float(step_losses["total"].detach()))

    print(f"step {step:2d}: total={loss_history[-1]:.3f} "
          f"rec={float(step_losses['reconstruction']):.3f} "
          f"align={float(step_losses['alignment']):.3f} "
          f"spatial={float(step_losses['spatial']):.3f}")


In [ ]:
with torch.no_grad():
    z_final = mini_model.embed(sample["modality_blocks"], sample["presence_mask"], sample["coords"])
print(f"Final embedding shape: {z_final.shape}  (expected [100, 32])")

# NOTE: with only 10 steps on a TINY 100-spot toy graph, loss can be noisy
# step-to-step (the spatial loss in particular is a sum over edges, so it
# scales with graph size/density and is not directly comparable across runs
# of different sizes). We compare the mean of the first 3 vs. last 3 steps
# instead of a single before/after point for a slightly more robust signal.
first_avg = sum(loss_history[:3]) / 3
last_avg = sum(loss_history[-3:]) / 3
print(f"Mean loss, first 3 steps: {first_avg:.3f} | last 3 steps: {last_avg:.3f}")
print("(Full-scale training with the real config -- 100 epochs, early stopping -- "
      "is needed to see a clean monotonic decrease; this toy loop is only a wiring check.)")


## Paper Results Comparison

The table below reproduces the paper's headline cohort-level numbers
(Table 2, from the SIR's `evaluation_protocol.reported_results`) for
reference. Your synthetic-data run above will **not** match these — see the
note in `evaluate.py`'s output and `data/README_data.md` for why.


In [ ]:
paper_results = {
    "dataset": "private 11-sample melanoma cohort (54,912 spots)",
    "method": "LATTICE, M5 (full multimodal)",
    "ARI": 0.329, "NMI": 0.450, "spatial_contiguity": 0.850,
    "silhouette": 0.417, "MUS": 0.803,
    "best_baseline_ARI (GraphST, M1)": 0.423,
}
print("Paper's claimed results (Table 2, LATTICE M5 vs. best ARI baseline):")
for k, v in paper_results.items():
    print(f"  {k}: {v}")
print()
print("To reproduce these results on YOUR OWN data, run:")
print("    python train.py --config configs/config.yaml")
print("    python evaluate.py --checkpoint checkpoints/best.pt --modality-level M5")
print("Then feed your outputs to ArXivist's Stage 6 Results Comparator.")


## What to do next

1. **Full training**: `python train.py --config configs/config.yaml`
2. **Evaluation**: `python evaluate.py --checkpoint checkpoints/best.pt`
3. **Compare results**: feed your results back to ArXivist's Results Comparator (Stage 6)

**Top implementation assumptions from the SIR (see `sir.json` for the full list):**
1. Reconstruction loss uses **Huber (δ=1.0)** per Appendix H, not the MSE shown in main-text Eq. 6 (confidence 0.6).
2. Cross-modal alignment uses only the **(Visium RNA, spatial ATAC)** pair per Appendix H, not all pairwise modality combinations (confidence 0.55).
3. The real evaluation cohort is **private/unreleasable**, so this repo trains on synthetic data matching the documented shapes (confidence 0.8 that this is unavoidable).
